# MATH840 Final project — model family: ARIMA

The ACF/PACF reading, an AICc grid, SARIMA on several training lengths, AutoARIMA, regression with SARIMA errors on three sets of weather regressors (ARIMAX), dynamic harmonic regression (DHR) on 1 to 12 weeks, and daily totals (with and without the weather) x the hourly profile.

Same data, same **20 rolling 48-hour windows** (one origin every midnight, 6–25 October), same scores as
`LUKIANETS_project_v2.ipynb`, judged on **day 2** (hours 25–48, the private leaderboard's hours). The last cell writes
`family_arima_bt.csv` and `family_arima_final.csv`; the main notebook merges them (Section 2.10) into one scoreboard, where the
combinations, the night rule and the final choice are made. Run it from top to bottom; the family notebooks are
independent of each other and can run at the same time. With `RESUME = True` (helpers cell) the models of an earlier
run are read back from the export and only new sections compute.

## 0. Setup

Download `train.csv` and `weather_forecast.csv` from Moodle. In Colab, the next cell asks you to upload
them. On your own machine, put them in the same folder as the notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

SURNAME = "LUKIANETS"   # <-- for your submission file name

if not SURNAME.strip():
    raise ValueError("Set SURNAME before running the rest.")

In [2]:
NEEDED = ["train.csv", "weather_forecast.csv"]

if not all(Path(f).exists() for f in NEEDED):
    try:
        from google.colab import files
        files.upload()                       # choose both files at once
    except ImportError:
        raise FileNotFoundError(f"Put {NEEDED} next to this notebook.")

data = pd.read_csv("train.csv", parse_dates=["ds"])
weather_fc = pd.read_csv("weather_forecast.csv", parse_dates=["ds"])

H = 48                                       # hours to forecast
M = 24                                       # the benchmark's season: one day
FUTURE = weather_fc["ds"]

assert len(FUTURE) == H and FUTURE.iloc[0] == data["ds"].iloc[-1] + pd.Timedelta(hours=1)
print(f"history: {len(data)} hours, {data['ds'].min()} to {data['ds'].max()}")
print(f"you are forecasting: {FUTURE.iloc[0]} to {FUTURE.iloc[-1]}")
data.tail()

history: 33480 hours, 2023-01-01 01:00:00 to 2026-10-27 00:00:00
you are forecasting: 2026-10-27 01:00:00 to 2026-10-29 00:00:00


,ds,y,cloud_cover,temperature,wind_speed,pressure,humidity,daylight,air_density
33475,2026-10-26 20:00:00,42.5,1.0,9,3,763,87,0,1.259
33476,2026-10-26 21:00:00,76.7,1.0,9,4,763,87,0,1.259
33477,2026-10-26 22:00:00,65.8,1.0,7,3,763,100,0,1.268
33478,2026-10-26 23:00:00,69.9,1.0,7,3,763,100,0,1.268
33479,2026-10-27 00:00:00,69.9,1.0,7,3,763,100,0,1.268


### The data

| Column | Meaning |
|---|---|
| `ds` | the hour |
| `y` | electricity generated in that hour; this is what you forecast |
| `cloud_cover` | from 0 (clear) to 1 (overcast), in steps of 0.25 |
| `temperature` | °C |
| `wind_speed` | m/s |
| `pressure` | mm Hg |
| `humidity` | % |
| `daylight` | 1 while the sun is above the horizon |
| `air_density` | kg/m³ |

`train.csv` has the weather as it was. `weather_forecast.csv` has a forecast for the 48 hours you
predict, of the kind a weather service issues. Its errors grow from the first day to the second.

## 1. How you are scored

Your file gets three scores on the 48 hidden hours, and lower is better for each. RMSE measures the
point forecast in the units of `y`, and MAPE measures it in percent of the actual value. The Winkler
score measures the 95% interval: its width, plus 40 times ($2/\alpha$) the distance by which the actual
value falls outside it.

The benchmark is a seasonal naive forecast with a daily season. Each hour of the next two days gets the
value of the same hour on the last day you have. Its 95% interval is the standard one for seasonal naive,
$\hat y \pm 1.96\,\hat\sigma\sqrt{k+1}$, where $k$ is the number of full days into the horizon.

The project description on Moodle explains the points and the leaderboard.

In [3]:
def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


def mape(actual, forecast) -> float:
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return float(np.mean(np.abs((actual - forecast) / actual)) * 100)


def winkler(actual, lower, upper, alpha: float = 0.05) -> float:
    actual, lower, upper = (np.asarray(a, float) for a in (actual, lower, upper))
    below = np.clip(lower - actual, 0, None)
    above = np.clip(actual - upper, 0, None)
    return float(np.mean((upper - lower) + 2 / alpha * (below + above)))


def score(actual, forecast, lower, upper) -> dict:
    inside = (np.asarray(actual) >= lower) & (np.asarray(actual) <= upper)
    return {"RMSE": rmse(actual, forecast), "MAPE": mape(actual, forecast),
            "Winkler": winkler(actual, lower, upper), "coverage": float(inside.mean())}


def snaive(history: np.ndarray, h: int = H, m: int = M):
    # The benchmark: repeat the last season, with the seasonal-naive 95% interval.
    history = np.asarray(history, float)
    point = np.array([history[-m + (i % m)] for i in range(h)])
    sigma = np.std(history[m:] - history[:-m], ddof=1)
    k = np.arange(h) // m
    half = 1.96 * sigma * np.sqrt(k + 1)
    return point, np.clip(point - half, 0, None), point + half

### The benchmark on a validation window

You cannot score a forecast on the hidden hours, so the cell below holds back the last 48 hours of the
history, forecasts them from the hours before, and scores the result. This is a single window. You can
decide how much more validation you need.

In [4]:
y = data["y"].to_numpy(float)
past, valid = y[:-H], y[-H:]

bench = snaive(past)
results = pd.DataFrame({"seasonal naive": score(valid, *bench)}).T
results

,RMSE,MAPE,Winkler,coverage
seasonal naive,120.404,51.088,568.464,1.0


## The protocol (shared with every notebook)

In [5]:
import itertools
import os
import time
import logging
os.environ["NIXTLA_ID_AS_COL"] = "true"
for name in ("transformers", "root", "cmdstanpy", "prophet", "lightning", "lightning.pytorch", "gluonts",
             "nixtla.nixtla_client", "pytorch_lightning", "neuralforecast"):
    logging.getLogger(name).setLevel(logging.ERROR)

from IPython.display import display
from scipy import signal, stats
from scipy.special import boxcox
from coreforecast.scalers import boxcox_lambda
from statsmodels.tsa.seasonal import MSTL as MSTL_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsforecast.arima import ARIMASummary, ndiffs, nsdiffs
from statsforecast.models import ARIMA, AutoARIMA, AutoETS, AutoTheta, MSTL
try:
    from dotenv import load_dotenv
    load_dotenv()                  # .env next to the notebook holds NIXTLA_API_KEY: never paste the key here
except ImportError:
    pass

dates = data["ds"].reset_index(drop=True)
FREQ = "h"
DAY = 24
WINDOWS = 20                       # rolling 48-hour windows: the SAME on the Mac and in Colab (merged later)
STEP = 24                          # hours between window origins: one origin every midnight (6-25 October)
PART = "day2"                      # the hours a choice is judged on: "day2" (hours 25-48, the private part) or "all"
LAMBDA = 0.45                      # Box-Cox lambda (Guerrero on the last 12 weeks, Section 2.2)
FM_CONTEXT = 2048                  # hours of history a foundation model reads (default)

T0 = time.time()
def elapsed():
    print(f"[{(time.time() - T0) / 60:.1f} min since the start of Section 2]")

In [6]:
NAN = lambda h: np.full(h, np.nan)
FORECASTERS, MIXES, POST, FAMILY, FINAL = {}, {}, {}, {}, {}
bt = None
BASE_COLS = ("window", "step", "actual", "daylight", "benchmark", "benchmark lo", "benchmark hi")
PURGE = -(-H // STEP) - 1          # neighbours this close share hours with a window (1 for 48-hour windows a day apart)


def fwd(x, kind):
    x = np.asarray(x, float)
    if kind == "log":
        return np.log(x)
    if kind == "bc":
        return boxcox(x, LAMBDA)
    return x


def inv(z, kind):
    z = np.asarray(z, float)
    if kind == "log":
        return np.exp(z)
    if kind == "bc":
        return np.power(np.maximum(LAMBDA * z + 1, 1e-12), 1 / LAMBDA)
    return z


def weather_features(df: pd.DataFrame) -> pd.DataFrame:
    # the weather as regressors, plus the physical interactions trees and linear models cannot find alone
    df = df.reset_index(drop=True)
    out = df[["daylight", "cloud_cover", "temperature", "wind_speed", "humidity", "pressure", "air_density"]].astype(float).copy()
    out["sun"] = out["daylight"] * (1 - out["cloud_cover"])         # clear sky while the sun is up
    out["humid_day"] = out["daylight"] * out["humidity"]             # humidity matters while the sun is up
    out["wind3"] = out["wind_speed"] ** 3 / 100                      # wind power grows with the cube of the speed
    # Where the sun is, from the daylight column alone (no outside data): how long the day is, how far into its
    # daylight hours an hour lies, a clear-sky curve peaking at solar noon, and the season. 0/1 daylight cannot say
    # that late-October days are shorter and the sun lower, which is what the models missed in window 1.
    ds = pd.to_datetime(df["ds"])
    day = ds.dt.normalize().to_numpy()
    light = out["daylight"].to_numpy()
    length = pd.Series(light).groupby(day).transform("sum").to_numpy()
    order = pd.Series(light).groupby(day).cumsum().to_numpy()
    position = np.where(light == 1, (order - 0.5) / np.maximum(length, 1), 0.0)
    out["day_length"] = length
    out["sun_position"] = position
    out["clear_sky"] = np.where(light == 1, np.sin(np.pi * position), 0.0)
    out["doy_sin"] = np.sin(2 * np.pi * ds.dt.dayofyear.to_numpy() / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * ds.dt.dayofyear.to_numpy() / 365.25)
    return out


FEAT_DF = weather_features(pd.concat([data.drop(columns="y"), weather_fc], ignore_index=True))   # history, then the forecast
FEAT_COLS = list(FEAT_DF.columns)
FEAT = FEAT_DF.to_numpy()
ALL_DATES = pd.concat([dates, FUTURE], ignore_index=True)
COV_COLS = ["humidity", "wind_speed", "cloud_cover", "temperature", "daylight", "humid_day", "wind3"]   # the weather
SUN_COLS = ["day_length", "sun_position", "clear_sky", "doy_sin", "doy_cos"]                           # the sun


def cov(cols, start, stop):
    # covariate rows (len(cols), stop - start) for the absolute hours start .. stop-1 (the forecast after the history)
    return FEAT_DF[cols].to_numpy(float)[start:stop].T


def _clean(out, h) -> dict:
    get = lambda *keys: next((np.asarray(out[k], float).ravel()[:h] for k in keys if k in out), NAN(h))
    return {"mean": get("mean"), "lo": get("lo", "lo-95"), "hi": get("hi", "hi-95")}


def on_scale(fit_predict, transform="log"):
    # fit_predict(x, end, h) sees fwd(y[:end]); its forecasts come back on the original scale
    def forecast(end, h):
        out = _clean(fit_predict(fwd(y[:end], transform), end, h), h)
        out = {k: inv(v, transform) for k, v in out.items()}
        out["mean"], out["lo"] = np.maximum(out["mean"], 0), np.maximum(out["lo"], 0)
        return out
    return forecast


def sf(make, transform="log", context=1344, level=True):
    # a statsforecast model: make() builds a fresh one, fitted on the last `context` hours
    def fit_predict(x, end, h):
        x = x[-context:] if context else x
        return make().forecast(y=x, h=h, level=[95]) if level else make().forecast(y=x, h=h)
    return on_scale(fit_predict, transform)


FORECASTERS["benchmark"] = lambda end, h: dict(zip(("mean", "lo", "hi"), snaive(y[:end], h)))


def window_ends(windows=None):
    # where each window ends: window 1 ends with the history, the others STEP hours apart before it
    return [len(y) - (k - 1) * STEP for k in range(windows or WINDOWS, 0, -1)]


def backtest(forecasters: dict, windows=None) -> pd.DataFrame:
    # every forecaster on every window, each window forecast from the hours before it only
    windows = windows or WINDOWS
    ends = window_ends(windows)
    frames = []
    for k, e in zip(range(windows, 0, -1), ends):
        p, lo, hi = snaive(y[:e - H])
        frames.append(pd.DataFrame({"window": k, "step": np.arange(1, H + 1), "actual": y[e - H:e],
                                    "daylight": data["daylight"].to_numpy()[e - H:e],
                                    "benchmark": p, "benchmark lo": lo, "benchmark hi": hi}))
    frame = pd.concat(frames, ignore_index=True)
    bench_rmse = np.mean([rmse(g["actual"], g["benchmark"]) for _, g in frame.groupby("window")])
    seconds = {}
    for name, f in forecasters.items():
        start = time.time()
        try:
            parts = [_clean(f(e - H, H), H) for e in ends]
            if any(np.isnan(p["mean"]).any() for p in parts):
                raise ValueError("NaN in the forecast")
        except Exception as err:
            print(f"  ✗ {name}: {type(err).__name__}: {str(err)[:200]}")
            continue
        FORECASTERS[name] = f
        for key, col in (("mean", name), ("lo", f"{name} lo"), ("hi", f"{name} hi")):
            frame[col] = np.concatenate([p[key] for p in parts])
        seconds[name] = time.time() - start
        r = np.mean([rmse(g["actual"], g[name]) for _, g in frame.groupby("window")])
        print(f"  ✓ {name:<46s} RMSE {r:8.1f}  (skill {r / bench_rmse:.3f})   {seconds[name]:.1f}s", flush=True)
    frame.attrs["seconds"] = seconds
    return frame


def add(bt, new):
    out = bt.drop(columns=[c for c in new.columns if c in bt.columns and c not in BASE_COLS])
    out = pd.concat([out, new[[c for c in new.columns if c not in out.columns]]], axis=1)
    out.attrs["seconds"] = {**bt.attrs.get("seconds", {}), **new.attrs.get("seconds", {})}
    return out


def forecasts_in(bt) -> list[str]:
    return [c for c in bt.columns if c not in BASE_COLS and not c.endswith((" lo", " hi"))] + ["benchmark"]


def _part(frame, part=None):
    # the rows of the judged hours: day 2 = steps 25-48 (the private part), day 1 = steps 1-24, all = 1-48
    part = part or PART
    if part == "day2":
        return frame[frame["step"] > DAY]
    if part == "day1":
        return frame[frame["step"] <= DAY]
    return frame


def _per_window(rows, f, lo=None, hi=None):
    # RMSE, MAPE (and Winkler, coverage) of forecast f in each window of `rows`, as arrays (fast: no groupby)
    _, ix = np.unique(rows["window"].to_numpy(), return_inverse=True)
    n = np.bincount(ix)
    a, f = rows["actual"].to_numpy(float), np.asarray(f, float)
    out = [np.sqrt(np.bincount(ix, (a - f) ** 2) / n), np.bincount(ix, np.abs((a - f) / a)) / n * 100]
    if lo is not None:
        lo, hi = np.asarray(lo, float), np.asarray(hi, float)
        miss = np.clip(lo - a, 0, None) + np.clip(a - hi, 0, None)
        out += [np.bincount(ix, (hi - lo) + 40 * miss) / n, np.bincount(ix, ((a >= lo) & (a <= hi)).astype(float)) / n]
    return out


def window_table(columns=None, per_window=False, part=None) -> pd.DataFrame:
    # mean over the windows of the template's scores on the judged hours; skill = score / benchmark's;
    # won = windows with the lowest RMSE; beat bench = windows with a lower RMSE than the seasonal naive
    columns = list(dict.fromkeys((forecasts_in(bt) if columns is None else list(columns)) + ["benchmark"]))
    rows = _part(bt, part)
    ids = np.unique(rows["window"].to_numpy())
    stats_, per = {}, {}
    for c in columns:
        has = rows[f"{c} lo"].notna().all()
        r, m, *wc = _per_window(rows, rows[c], rows[f"{c} lo"] if has else None, rows[f"{c} hi"] if has else None)
        stats_[c] = {"RMSE": r.mean(), "MAPE": m.mean(), "Winkler": wc[0].mean() if has else np.nan,
                     "coverage": wc[1].mean() if has else np.nan}
        per[c] = pd.Series(r, index=ids)
    tab = pd.DataFrame(stats_).T
    b = tab.loc["benchmark"]
    tab["RMSE skill"], tab["MAPE skill"] = tab["RMSE"] / b["RMSE"], tab["MAPE"] / b["MAPE"]
    tab["point skill"] = (tab["RMSE skill"] + tab["MAPE skill"]) / 2
    tab["Winkler skill"] = tab["Winkler"] / b["Winkler"]
    w_rmse = pd.DataFrame(per)
    tab["won"] = [int((w_rmse.idxmin(axis=1) == c).sum()) for c in tab.index]
    tab["beat bench"] = [int((w_rmse[c] < w_rmse["benchmark"]).sum()) for c in tab.index]
    tab["seconds"] = [bt.attrs.get("seconds", {}).get(c, np.nan) for c in tab.index]
    tab = tab.sort_values("point skill")
    if per_window:
        return tab, (w_rmse.T / w_rmse["benchmark"]).loc[tab.index].round(2)
    return tab


def run(family: str, forecasters: dict, rerun=False) -> pd.DataFrame:
    global bt
    todo = {n: f for n, f in forecasters.items() if rerun or bt is None or n not in bt.columns}
    if todo:
        print(f"{family}: {len(todo)} forecaster(s) x {WINDOWS} windows")
        new = backtest(todo)
        bt = new if bt is None else add(bt, new)
    for n in forecasters:
        if n in bt.columns:
            FAMILY[n] = family
            if n in todo:
                FINAL.pop(n, None)            # re-run: its 48-hour forecast is computed again at the export
    return window_table([n for n in forecasters if n in bt.columns])


def best_of(family: str) -> str:
    names = [n for n, f in FAMILY.items() if f == family and n in bt.columns]
    return window_table(names).drop(index="benchmark")["point skill"].idxmin()


def combine(names, label, how="mean", interval_from=None):
    # the members' mean (or median); the interval: the members' mean interval, or -- interval_from -- that member's
    # own interval stretched to contain the combined point (for members without an interval of their own)
    names = [n for n in dict.fromkeys(names) if n in bt.columns]
    MIXES[label], FAMILY[label] = (names, how, interval_from), "combination"
    FINAL.pop(label, None)
    agg = (lambda cols, **kw: bt[cols].median(axis=1, **kw)) if how == "median" else (lambda cols, **kw: bt[cols].mean(axis=1, **kw))
    bt[label] = agg(names)
    if interval_from:
        bt[f"{label} lo"] = np.minimum(bt[f"{interval_from} lo"], bt[label])
        bt[f"{label} hi"] = np.maximum(bt[f"{interval_from} hi"], bt[label])
    else:
        bt[f"{label} lo"] = agg([f"{n} lo" for n in names], skipna=False)
        bt[f"{label} hi"] = agg([f"{n} hi" for n in names], skipna=False)


def others(w):
    # the windows that share no hour with window w: w and its PURGE neighbours on each side are left out
    return bt[(bt["window"] - w).abs() > PURGE]


def _radius(actual, point, daylight, recipe):
    # the 95% conformal radius from past errors: additive, or additive split day / night
    err = np.abs(np.asarray(actual, float) - np.asarray(point, float))
    if recipe == "conformal day/night":
        return {d: np.quantile(err[np.asarray(daylight) == d], 0.95) for d in (0, 1)}
    return np.quantile(err, 0.95)


def _bounds(point, daylight, r, recipe):
    point = np.asarray(point, float)
    rr = np.where(np.asarray(daylight) == 1, r[1], r[0]) if recipe == "conformal day/night" else r
    return np.maximum(point - rr, 0), point + rr


SCALES = np.round(np.arange(0.5, 1.501, 0.05), 2)


def _scaled(point, lo, hi, c):
    # the interval stretched (c > 1) or shrunk (c < 1) around the point forecast
    point, lo, hi = (np.asarray(v, float) for v in (point, lo, hi))
    return np.maximum(point - c * (point - lo), 0), point + c * (hi - point)


def _best_scale(rows, label):
    # the width factor with the lowest mean Winkler on `rows`
    p, lo, hi = rows[label], rows[f"{label} lo"], rows[f"{label} hi"]
    return min(SCALES, key=lambda c: _per_window(rows, p, *_scaled(p, lo, hi, c))[2].mean())


RECIPES = ["conformal", "conformal day/night"]


def interval_check(label: str, part=None) -> pd.DataFrame:
    # every recipe judged on windows it did not see: radius / width factor from the windows sharing no hour with it
    rows = []
    for w in sorted(bt["window"].unique()):
        g, o = _part(bt[bt["window"] == w], part), _part(others(w), part)
        recipes = {rec: _bounds(g[label], g["daylight"], _radius(o["actual"], o[label], o["daylight"], rec), rec) for rec in RECIPES}
        if g[f"{label} lo"].notna().all():
            recipes["own interval"] = (g[f"{label} lo"].to_numpy(), g[f"{label} hi"].to_numpy())
            recipes["own interval x c"] = _scaled(g[label], g[f"{label} lo"], g[f"{label} hi"], _best_scale(o, label))
        for rec, (lo, hi) in recipes.items():
            s = score(g["actual"], g[label], lo, hi)
            rows.append({"recipe": rec, "window": w, "Winkler": s["Winkler"], "coverage": s["coverage"],
                         "width": float(np.mean(np.asarray(hi) - np.asarray(lo)))})
    return pd.DataFrame(rows).groupby("recipe")[["Winkler", "coverage", "width"]].mean().sort_values("Winkler")


SHADE_FACTORS = np.round(np.arange(0.1, 1.001, 0.05), 2)
SHADE_LEVELS = (50, 100, 150, 200, 300)


def _shade(point, a, level):
    # the night rule: forecasts below `level` multiplied by a (MAPE punishes over-forecasting small values hardest)
    point = np.asarray(point, float)
    return np.where(point < level, a * point, point)


def _tune_post(frame, base, factors=SHADE_FACTORS):
    # the night rule (a, level) with the lowest RMSE skill + MAPE skill on the judged hours of `frame`
    # (base: a Series on frame's index); returns a, level and the RMSE skill + MAPE skill reached
    rows = _part(frame)
    b = base[rows.index].to_numpy()
    br, bm = _per_window(rows, rows["benchmark"])
    obj = lambda a, level: sum(v.mean() / ref.mean() for v, ref in zip(_per_window(rows, _shade(b, a, level)), (br, bm)))
    a, level = min(((a, L) for a in factors for L in SHADE_LEVELS), key=lambda p: obj(*p))
    return float(a), int(level), float(obj(a, level))


def postprocess(day_label, night_label=None, name=None, shade=True) -> str:
    # 1) the day model in daylight hours; at night the same model or its log-scale version, whichever scores
    #    better (night_label fixes it);
    # 2) the night rule: forecasts below `level` multiplied by a (shade=False: no rule).
    # The interval: the one of the model used at each hour, stretched to include the point; its width is judged in
    # interval_check like any model's. The night model, a and level are tuned on the windows sharing no hour with
    # the scored one (purged leave-one-window-out), so bt holds an honest score; the submission uses them tuned on
    # all windows.
    log = f"{day_label} log"
    nights = [night_label] if night_label else [day_label] + ([log] if log in bt.columns else [])
    name = name or f"{day_label} [{'post' if shade else 'hybrid'}]"
    factors = SHADE_FACTORS if shade else (1.0,)
    night = bt["daylight"].to_numpy() == 0
    pick = lambda n, s: pd.Series(np.where(night, bt[n + s], bt[day_label + s]), index=bt.index)

    def tune(frame):
        # [night model, a, level]: the night model whose tuned rule reaches the lowest RMSE skill + MAPE skill
        fits = {n: _tune_post(frame, pick(n, "")[frame.index], factors) for n in nights}
        n = min(fits, key=lambda k: fits[k][2])
        return [n, *fits[n][:2]]

    point, plo, phi = (pd.Series(np.nan, index=bt.index) for _ in range(3))
    chosen = []
    for w in sorted(bt["window"].unique()):
        n, a, level = tune(others(w))
        chosen.append(("log" if n == log else "same" if n == day_label else n, a, level))
        idx = bt.index[bt["window"] == w]
        point[idx] = _shade(pick(n, "")[idx], a, level)
        plo[idx] = np.minimum(pick(n, " lo")[idx], point[idx])
        phi[idx] = np.maximum(pick(n, " hi")[idx], point[idx])
    bt[name], bt[f"{name} lo"], bt[f"{name} hi"] = point, plo, phi
    n, a, level = tune(bt)
    POST[name], FAMILY[name] = {"day": day_label, "night": n, "a": a, "level": level}, "post-processed"
    FINAL.pop(name, None)
    print(f"{name}: night = {n}; submission rule a = {a}, level = {level}; held-out windows chose (night, a, level) "
          f"{pd.Series([str(x) for x in chosen]).value_counts().to_dict()}")
    return name


def from_all_of_y(label: str) -> dict:
    # every forecaster inside `label` once more on the WHOLE history (cached), averaged / post-processed as in bt
    if label not in FINAL:
        if label in MIXES:
            names, how, interval_from = MIXES[label]
            parts = {n: from_all_of_y(n) for n in names}
            agg = np.median if how == "median" else np.mean
            point = agg([p["mean"] for p in parts.values()], axis=0)
            if interval_from:
                lo, hi = np.minimum(parts[interval_from]["lo"], point), np.maximum(parts[interval_from]["hi"], point)
            else:
                lo, hi = (agg([p[k] for p in parts.values()], axis=0) for k in ("lo", "hi"))
            FINAL[label] = {"mean": point, "lo": lo, "hi": hi}
        elif label in POST:
            p = POST[label]
            d_, n_ = from_all_of_y(p["day"]), from_all_of_y(p["night"])
            night = weather_fc["daylight"].to_numpy()[:H] == 0
            base, lo, hi = (np.where(night, n_[k], d_[k]) for k in ("mean", "lo", "hi"))
            point = _shade(base, p["a"], p["level"])
            FINAL[label] = {"mean": point, "lo": np.minimum(lo, point), "hi": np.maximum(hi, point)}
        else:
            FINAL[label] = _clean(FORECASTERS[label](len(y), H), H)
    return FINAL[label]


def final_forecast(label: str, interval: str):
    # the 48 hours from the whole history; a conformal radius or width factor is fitted for each day of the horizon
    # separately (on all windows' day-1 errors for hours 1-24, on their day-2 errors for hours 25-48)
    fc = from_all_of_y(label)
    point = np.maximum(fc["mean"], 0)
    lo, hi = np.array(fc["lo"], float), np.array(fc["hi"], float)
    light = weather_fc["daylight"].to_numpy()[:H]
    for part, sl in (("day1", slice(0, DAY)), ("day2", slice(DAY, H))):
        rows = _part(bt, part)
        if interval == "own interval x c":
            lo[sl], hi[sl] = _scaled(point[sl], fc["lo"][sl], fc["hi"][sl], _best_scale(rows, label))
        elif interval in RECIPES:
            r = _radius(rows["actual"], rows[label], rows["daylight"], interval)
            lo[sl], hi[sl] = _bounds(point[sl], light[sl], r, interval)
        elif interval != "own interval":
            raise ValueError("interval: 'own interval', 'own interval x c' or one of RECIPES")
    if np.isnan(lo).any():
        raise ValueError(f"{label} has no interval of its own: use a conformal recipe")
    return point, np.minimum(np.maximum(lo, 0), point), np.maximum(hi, point)


def plot_validation(names, window=1, show=72):
    g = bt[bt["window"] == window]
    end = len(y) - (window - 1) * STEP
    x = dates.iloc[end - H:end]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(dates.iloc[end - H - show:end - H], y[end - H - show:end - H], color="grey", lw=0.8, label="history")
    ax.plot(x, g["actual"], color="black", lw=1.6, label="actual")
    ax.plot(x, g["benchmark"], color="grey", ls="--", lw=1.2, label="seasonal naive")
    ax.axvline(x.iloc[DAY - 1], color="grey", ls=":", lw=1)
    for n in names:
        ax.plot(x, g[n], lw=1.4, label=f"{n} (RMSE {rmse(g['actual'], g[n]):.1f})")
        if len(names) == 1 and g[f"{n} lo"].notna().all():
            ax.fill_between(x, g[f"{n} lo"], g[f"{n} hi"], alpha=0.2, label="95% interval")
    ax.set_title(f"window {window}" + (" (the template's validation window)" if window == 1 else "") + ": day 1 | day 2")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()


RESULTS_DIR = "results"   # the exported model tables (<prefix>_bt.csv, <prefix>_final.csv) live here


def export(prefix: str):
    # the rolling-window table and each model's 48-hour forecast from the whole history, for merging elsewhere
    names = [n for n in forecasts_in(bt) if n != "benchmark" and n not in MIXES and n not in POST]
    Path(RESULTS_DIR).mkdir(exist_ok=True)
    bt[list(BASE_COLS) + [c for n in names for c in (n, f"{n} lo", f"{n} hi")]].to_csv(f"{RESULTS_DIR}/{prefix}_bt.csv", index=False)
    rows = []
    for n in names:
        try:
            fc = from_all_of_y(n)
        except Exception as err:
            print(f"  ✗ final forecast of {n}: {type(err).__name__}: {str(err)[:120]}")
            continue
        rows += [{"model": n, "family": FAMILY.get(n, "?"), "step": i + 1, "mean": fc["mean"][i], "lo": fc["lo"][i],
                  "hi": fc["hi"][i]} for i in range(H)]
    pd.DataFrame(rows).to_csv(f"{RESULTS_DIR}/{prefix}_final.csv", index=False)
    print(f"saved {RESULTS_DIR}/{prefix}_bt.csv ({len(names)} models) and {RESULTS_DIR}/{prefix}_final.csv")


def merge(prefix: str):
    # adds another notebook's exported models to bt, FAMILY and FINAL (the same windows are checked)
    global bt
    other = pd.read_csv(f"{RESULTS_DIR}/{prefix}_bt.csv")
    assert len(other) == len(bt) and (other["window"].to_numpy() == bt["window"].to_numpy()).all() \
        and np.allclose(other["actual"], bt["actual"]), "not the same windows: WINDOWS and STEP must match"
    new = [c for c in other.columns if c not in BASE_COLS]
    seconds = bt.attrs.get("seconds", {})
    bt = pd.concat([bt.drop(columns=[c for c in new if c in bt.columns]), other[new]], axis=1)
    bt.attrs["seconds"] = seconds
    fin = pd.read_csv(f"{RESULTS_DIR}/{prefix}_final.csv")
    for n, g in fin.groupby("model"):
        g = g.sort_values("step")
        FINAL[n] = {k: g[k].to_numpy(float) for k in ("mean", "lo", "hi")}
        FAMILY[n] = g["family"].iloc[0]
    print(f"merged {fin['model'].nunique()} models from {prefix}")

In [7]:
# Helpers used by the family notebooks
import os

RESUME = True       # <-- True: the models already in this notebook's export (an earlier run) are read back, only new
                    #     sections compute; False: everything from scratch
CONTEXTS = {"1w": 168, "2w": 336, "4w": 672, "8w": 1344, "12w": 2016, "26w": 4368}   # training lengths (hours)
LENGTH_ORDER = ["3d", "5d", "1w", "10d", "2w", "3w", "4w", "8w", "12w", "26w",
                "14d", "28d", "56d", "60d", "90d", "120d", "180d", "365d", "730d", "1095d"]


def resume(prefix):
    # RESUME: read back the models of this notebook's earlier export, so run() computes only the new ones
    global bt
    if RESUME and os.path.exists(f"{RESULTS_DIR}/{prefix}_bt.csv"):
        if bt is None:
            bt = pd.read_csv(f"{RESULTS_DIR}/{prefix}_bt.csv")[list(BASE_COLS)]
        merge(prefix)


def sf_any(make, transform="log", context=1344):
    # a statsforecast model with its 95% interval when it has one, the point forecast alone when it has not
    def fit_predict(x, end, h):
        x = x[-context:] if context else x
        try:
            return make().forecast(y=x, h=h, level=[95])
        except Exception:
            return make().forecast(y=x, h=h)
    return on_scale(fit_predict, transform)


def grid_view(table, value="point skill"):
    # "name, training length" rows -> a model x training-length table
    t = table.drop(index="benchmark").copy()
    t = t[[", " in n for n in t.index]]
    t["model"] = [n.rsplit(", ", 1)[0] for n in t.index]
    t["training"] = [n.rsplit(", ", 1)[1] for n in t.index]
    order = [c for c in LENGTH_ORDER if c in set(t["training"])]
    return t.pivot(index="model", columns="training", values=value)[order]


def show(table, title):
    print(title)
    display(table[["RMSE skill", "MAPE skill", "point skill", "Winkler skill", "coverage", "beat bench", "seconds"]].head(15).round(3))


# Regressors of the linear models: the weather + the sun (the set of the best DHR). Cloud-based interactions
# (clear-sky curve x (1 - cloud)) were tried and made linear models worse: cloud cover acts non-linearly here (EDA).
X_REG = ["humidity", "wind_speed", "cloud_cover", "temperature", "daylight", "humid_day", "wind3", "day_length", "clear_sky"]
X_DAILY = ["humid_day", "wind3", "temperature", "day_length"]          # daily means of these: a day's weather


def _try_forecast(model, y_, h, X=None, X_future=None):
    # the model's 95% interval when it has one, the point forecast alone when it has not
    kw = {} if X is None else {"X": X, "X_future": X_future}
    try:
        return model.forecast(y=y_, h=h, level=[95], **kw)
    except Exception:
        return model.forecast(y=y_, h=h, **kw)


def _varying(X, Xf):
    # drops the columns constant on the training rows (a regression cannot use them; e.g. daylight at 3 a.m.)
    keep = np.asarray(X).std(axis=0) > 1e-9
    return (X[:, keep], Xf[:, keep]) if keep.any() else (None, None)


def by_hour(make, transform="log", days=28, xcols=None):
    # one series per hour of the day: the last `days` values of each of the 24 hours form a daily series, forecast
    # 1 and 2 days ahead by make(); with xcols, that hour's weather as regressors (models that take X)
    def fit_predict(x, end, h):
        n = min(days, end // DAY)
        t = np.arange(end - n * DAY, end).reshape(n, DAY)                # rows: days; columns: the hours of the day
        steps = -(-h // DAY)
        tf = np.arange(end, end + steps * DAY).reshape(steps, DAY)
        F = FEAT_DF[xcols].to_numpy(float) if xcols else None
        out = {k: np.full(steps * DAY, np.nan) for k in ("mean", "lo", "hi")}
        for p in range(DAY):
            X, Xf = _varying(F[t[:, p]], F[tf[:, p]]) if xcols else (None, None)
            fc = _try_forecast(make(), np.asarray(x[t[:, p]], float), steps, X, Xf)
            for k, key in (("mean", "mean"), ("lo", "lo-95"), ("hi", "hi-95")):
                if key in fc:
                    out[k][p::DAY] = np.asarray(fc[key], float)[:steps]
        return {k: v[:h] for k, v in out.items()}
    return on_scale(fit_predict, transform)


def top_down(make, days=56, profile_days=7, transform="log", xcols=None):
    # the day's total (sum of its 24 hours) forecast by make() from the last `days` daily totals (on `transform`),
    # spread over the hours with the mean profile -- each hour's share of its day -- of the last `profile_days` days;
    # with xcols, the daily means of those columns as regressors (the forecast days' weather is known)
    def fit_predict(x, end, h):                                           # x: the original scale
        n = min(days, end // DAY)
        t = np.arange(end - n * DAY, end)
        block = np.asarray(x[t], float).reshape(n, DAY)
        recent = block[-profile_days:]
        share = (recent / recent.sum(axis=1, keepdims=True)).mean(axis=0)
        steps = -(-h // DAY)
        X = Xf = None
        if xcols:
            F = FEAT_DF[xcols].to_numpy(float)
            X = F[t].reshape(n, DAY, -1).mean(axis=1)
            Xf = F[np.arange(end, end + steps * DAY)].reshape(steps, DAY, -1).mean(axis=1)
            X, Xf = _varying(X, Xf)
        fc = _try_forecast(make(), fwd(block.sum(axis=1), transform), steps, X, Xf)
        return {k: np.concatenate([inv(np.asarray(fc[key], float)[d], transform) * share for d in range(steps)])[:h]
                for k, key in (("mean", "mean"), ("lo", "lo-95"), ("hi", "hi-95")) if key in fc}
    return on_scale(fit_predict, None)


def two_stage(make, xcols=X_REG, transform="log", context=2016, alpha=30.0):
    # regression with ETS (or other) errors in two stages: (1) ridge regression of the transformed series on the hour
    # of the day (24 dummies, not penalised) and the standardised weather, on the training window; (2) make() -- a
    # non-seasonal model -- on the DAILY MEANS of its residuals (what the weather does not explain: a level that moves),
    # forecast 1 and 2 days ahead and added to every hour of those days. (On the hourly residuals the last night's
    # residual was carried into tomorrow's midday and multiplied the peak.)
    def fit_predict(x, end, h):
        n = min(context, end) // DAY * DAY
        t, tf = np.arange(end - n, end), np.arange(end, end + h)
        hours = ALL_DATES.dt.hour.to_numpy()
        F = FEAT_DF[xcols].to_numpy(float)
        mu, sd = F[t].mean(axis=0), F[t].std(axis=0) + 1e-9
        design = lambda r: np.column_stack([(hours[r][:, None] == np.arange(DAY)).astype(float), (F[r] - mu) / sd])
        X, Xf = design(t), design(tf)
        z = np.asarray(x[t], float)
        beta = np.linalg.solve(X.T @ X + np.diag(np.r_[np.zeros(DAY), np.full(len(xcols), alpha)]), X.T @ z)
        resid = z - X @ beta
        steps = -(-h // DAY)
        fc = _try_forecast(make(), resid.reshape(-1, DAY).mean(axis=1), steps)
        reg = Xf @ beta
        spread = np.quantile(np.abs(resid - np.repeat(resid.reshape(-1, DAY).mean(axis=1), DAY)), 0.95)
        out = {"mean": reg + np.repeat(np.asarray(fc["mean"], float), DAY)[:h]}
        if "lo-95" in fc:   # the daily level's interval widened by the hours' spread around their day's mean
            out["lo"] = reg + np.repeat(np.asarray(fc["lo-95"], float), DAY)[:h] - spread
            out["hi"] = reg + np.repeat(np.asarray(fc["hi-95"], float), DAY)[:h] + spread
        return out
    return on_scale(fit_predict, transform)


def sf_x(make, xcols, transform="log", context=1344):
    # a statsforecast model that takes regressors (X, X_future): the weather of the training hours and of the horizon
    def fit_predict(x, end, h):
        t0 = max(0, end - context)
        F = FEAT_DF[xcols].to_numpy(float)
        return _try_forecast(make(), np.asarray(x[t0:], float), h, F[t0:end], F[end:end + h])
    return on_scale(fit_predict, transform)

In [8]:
resume("family_arima")      # reads back an earlier run when RESUME = True (see the cell above)

merged 52 models from family_arima


## 1. Differencing, and the ACF / PACF reading (Labs 6–7)

On the log scale over the last 8 weeks the tests give one seasonal difference (D = 1) and no first difference.
The ACF decays geometrically and the PACF cuts off after lag 2 → **AR(2)**; after the seasonal difference the ACF has
one negative spike at lag 24 and the PACF decays at 25, 49, 73 → **seasonal MA(1)**. Hand pick
**SARIMA(2,0,0)(0,1,1)[24]**; the alternative reading (P, Q = 2–3) is in the grid.

In [9]:
ARIMA_M = DAY
x8 = np.log(y[-1344:])
D = int(nsdiffs(x8, period=ARIMA_M))
z = x8[ARIMA_M:] - x8[:-ARIMA_M] if D else x8.copy()
d = int(ndiffs(z))
print(f"log scale, last 8 weeks: D = {D}, d = {d}")
AUTO_ARIMA_KW = {"approximation": True}
X_SETS = {                                                    # regressor sets for ARIMAX / DHR
    "weather":       ["humidity", "wind_speed", "cloud_cover", "temperature", "daylight", "humid_day", "wind3"],
    "weather + sun": ["humidity", "wind_speed", "cloud_cover", "temperature", "daylight", "humid_day", "wind3",
                      "day_length", "clear_sky"],
    "humidity + wind + sun": ["humidity", "humid_day", "wind_speed", "wind3", "clear_sky", "day_length"],
}


def fourier(t, m, K):
    t = np.asarray(t, float)
    return np.column_stack([f(2 * np.pi * k * t / m) for k in range(1, K + 1) for f in (np.sin, np.cos)])


def regressors(t, K=None, xcols=None):
    blocks = [fourier(t, m, k) for m, k in (K or {}).items() if k]
    if xcols:
        blocks.append(FEAT_DF[xcols].to_numpy(float)[np.asarray(t, int)])
    return np.column_stack(blocks) if blocks else None


def _arima_fit(x, t0, order=None, seasonal=(0, 0, 0), fourier_K=None, xcols=None, seasonal_search=True):
    X = regressors(np.arange(t0, t0 + len(x)), fourier_K, xcols)
    if order is None:
        model = AutoARIMA(season_length=ARIMA_M, seasonal=seasonal_search, **AUTO_ARIMA_KW)
    else:
        model = ARIMA(order=tuple(order), seasonal_order=tuple(seasonal), season_length=ARIMA_M)
    return model.fit(y=x, X=X)


def arima(transform="log", context=1344, **spec):
    def fit_predict(x, end, h):
        t0 = max(0, end - context)
        model = _arima_fit(x[t0:], t0, **spec)
        return model.predict(h=h, X=regressors(np.arange(end, end + h), spec.get("fourier_K"), spec.get("xcols")), level=[95])
    return on_scale(fit_predict, transform)


def arima_info(transform="log", context=1344, **spec) -> dict:
    x = fwd(past, transform)
    t0 = len(x) - context
    fit = _arima_fit(x[t0:], t0, **spec)
    p_, q_, P_, Q_ = fit.model_["arma"][:4]
    resid = pd.Series(fit.model_["residuals"])
    return {"fitted on past": str(ARIMASummary(fit.model_)).strip(), "AICc": fit.model_["aicc"],
            "LB p (48)": acorr_ljungbox(resid, lags=[48], model_df=p_ + q_ + P_ + Q_)["lb_pvalue"].iloc[0]}


GRID = {f"({p_},{d},{q_})({P_},{D},{Q_})": {"order": (p_, d, q_), "seasonal": (P_, D, Q_)}
        for p_ in (1, 2, 3) for q_ in (0, 1) for P_, Q_ in [(0, 1), (1, 1), (0, 2), (1, 0), (2, 2), (2, 3), (3, 2), (3, 3)]}
screen = {}
for name, spec in GRID.items():
    try:
        screen[name] = arima_info(**spec)
    except Exception as err:
        screen[name] = {"fitted on past": f"failed: {type(err).__name__}"}
screen = pd.DataFrame(screen).T.sort_values("AICc")
print(f"{len(GRID)} candidates by AICc on the last 8 weeks of `past` (log scale):")
display(screen.head(12))
elapsed()

log scale, last 8 weeks: D = 1, d = 0


48 candidates by AICc on the last 8 weeks of `past` (log scale):


,fitted on past,AICc,LB p (48)
"(1,0,1)(3,1,2)","ARIMA(1,0,1)(3,1,2)[24]",590.639,0.0
"(1,0,1)(3,1,3)","ARIMA(1,0,1)(3,1,3)[24]",593.548,0.0
"(1,0,1)(2,1,3)","ARIMA(1,0,1)(2,1,3)[24]",606.218,0.0
"(3,0,0)(2,1,3)","ARIMA(3,0,0)(2,1,3)[24]",611.057,0.0
"(2,0,0)(2,1,3)","ARIMA(2,0,0)(2,1,3)[24]",615.428,0.0
"(1,0,1)(0,1,1)","ARIMA(1,0,1)(0,1,1)[24]",617.751,0.0
"(1,0,1)(2,1,2)","ARIMA(1,0,1)(2,1,2)[24]",617.961,0.0
"(3,0,1)(0,1,1)","ARIMA(3,0,1)(0,1,1)[24]",618.717,0.0
"(2,0,1)(0,1,1)","ARIMA(2,0,1)(0,1,1)[24]",618.986,0.0
"(1,0,1)(0,1,2)","ARIMA(1,0,1)(0,1,2)[24]",619.601,0.0


[2.9 min since the start of Section 2]


## 2. SARIMA on several training lengths, and AutoARIMA

In [10]:
HAND = [f"(2,{d},0)(0,{D},1)", f"(1,{d},0)(0,{D},1)", f"(2,{d},1)(0,{D},1)", f"(2,{d},0)(2,{D},2)"]
by_aicc = [n for n in screen.index if pd.notna(screen.loc[n, "AICc"])]
SELECTED = list(dict.fromkeys(HAND + by_aicc[:4]))
print("backtested:", SELECTED)
SARIMA = {f"SARIMA{n} log, {c}": arima(context=CONTEXTS[c], **GRID[n]) for n in SELECTED for c in ("2w", "4w", "8w")}
SARIMA.update({f"AutoARIMA {kind}, {c}": arima(transform=kind, context=CONTEXTS[c], order=None)
               for kind in ("log", "bc") for c in ("4w", "8w")})
sarima_table = run("ARIMA", SARIMA)
display(grid_view(sarima_table).round(3))
show(sarima_table, "SARIMA / AutoARIMA (day 2):")
elapsed()

backtested: ['(2,0,0)(0,1,1)', '(1,0,0)(0,1,1)', '(2,0,1)(0,1,1)', '(2,0,0)(2,1,2)', '(1,0,1)(3,1,2)', '(1,0,1)(3,1,3)', '(1,0,1)(2,1,3)', '(3,0,0)(2,1,3)']


training,2w,4w,8w
model,,,
AutoARIMA bc,NaN,0.721,0.779
AutoARIMA log,NaN,0.743,0.751
"SARIMA(1,0,0)(0,1,1) log",0.749,0.732,0.692
"SARIMA(1,0,1)(2,1,3) log",0.690,0.652,0.674
"SARIMA(1,0,1)(3,1,2) log",0.672,0.655,0.679
"SARIMA(1,0,1)(3,1,3) log",0.675,0.674,0.670
"SARIMA(2,0,0)(0,1,1) log",0.685,0.656,0.666
"SARIMA(2,0,0)(2,1,2) log",0.675,0.660,0.671
"SARIMA(2,0,1)(0,1,1) log",0.690,0.665,0.665


SARIMA / AutoARIMA (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"SARIMA(3,0,0)(2,1,3) log, 4w",0.946,0.349,0.647,4.232e+01,0.908,11,NaN
"SARIMA(1,0,1)(2,1,3) log, 4w",0.956,0.348,0.652,1.119e+00,0.865,12,NaN
"SARIMA(1,0,1)(3,1,2) log, 4w",0.994,0.316,0.655,1.036e+00,0.875,10,NaN
"SARIMA(2,0,0)(0,1,1) log, 4w",0.927,0.385,0.656,1.205e+00,0.860,12,NaN
"SARIMA(2,0,0)(2,1,2) log, 4w",0.966,0.354,0.660,1.182e+00,0.869,9,NaN
"SARIMA(2,0,1)(0,1,1) log, 8w",0.938,0.391,0.665,9.880e-01,0.846,11,NaN
"SARIMA(2,0,1)(0,1,1) log, 4w",0.943,0.387,0.665,1.223e+00,0.856,11,NaN
"SARIMA(2,0,0)(0,1,1) log, 8w",0.941,0.392,0.666,9.870e-01,0.846,11,NaN
"SARIMA(1,0,1)(3,1,3) log, 8w",0.952,0.388,0.670,1.026e+00,0.840,11,NaN
"SARIMA(2,0,0)(2,1,2) log, 8w",0.944,0.398,0.671,1.012e+00,0.848,10,NaN


[2.9 min since the start of Section 2]


## 3. The weather as regressors: ARIMAX and dynamic harmonic regression

Regression with SARIMA errors on three regressor sets, on log and Box-Cox, for the two best (order, length); and DHR
— Fourier terms of the daily season (no weekly season: EDA) + regressors + non-seasonal ARIMA errors.

In [11]:
plain = window_table([n for n in sarima_table.index if n.startswith("SARIMA")]).drop(index="benchmark")
ARIMAX = {}
for n in plain.index[:2]:
    order_name, ctx_name = n.removeprefix("SARIMA").split(" log, ")
    for xs, cols in X_SETS.items():
        for kind in ("log", "bc"):
            ARIMAX[f"SARIMA{order_name} + {xs} {kind}, {ctx_name}"] = arima(transform=kind, context=CONTEXTS[ctx_name],
                                                                            xcols=cols, **GRID[order_name])
for K in (3, 5, 10):
    for xs in ("weather", "weather + sun"):
        for c in ("4w", "8w"):
            ARIMAX[f"DHR K={K} + {xs} log, {c}"] = arima(context=CONTEXTS[c], order=None, seasonal_search=False,
                                                         fourier_K={DAY: K}, xcols=X_SETS[xs])
show(run("ARIMAX / DHR", ARIMAX), "ARIMAX and DHR (day 2):")
elapsed()

ARIMAX and DHR (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"SARIMA(1,0,1)(2,1,3) + weather + sun log, 4w",0.817,0.312,0.564,9.920e-01,0.877,14,NaN
"DHR K=10 + weather + sun log, 4w",0.736,0.413,0.575,8.870e-01,0.867,19,NaN
"DHR K=5 + weather + sun log, 4w",0.715,0.441,0.578,8.950e-01,0.865,18,NaN
"DHR K=3 + weather + sun log, 4w",0.732,0.430,0.581,8.460e-01,0.860,18,NaN
"SARIMA(3,0,0)(2,1,3) + humidity + wind + sun log, 4w",0.862,0.309,0.586,1.852e+02,0.923,12,NaN
"SARIMA(1,0,1)(2,1,3) + humidity + wind + sun log, 4w",0.859,0.314,0.587,1.009e+00,0.881,13,NaN
"DHR K=10 + weather + sun log, 8w",0.727,0.451,0.589,9.010e-01,0.831,16,NaN
"DHR K=5 + weather + sun log, 8w",0.709,0.472,0.591,9.020e-01,0.831,18,NaN
"DHR K=3 + weather + sun log, 8w",0.712,0.470,0.591,8.570e-01,0.825,18,NaN
"SARIMA(3,0,0)(2,1,3) + weather log, 4w",0.842,0.343,0.592,7.934e+08,0.912,12,NaN


[2.9 min since the start of Section 2]


## 4. Shorter DHR, the weather × sun interactions, one ARIMAX per hour, daily totals

DHR + the weather + the sun was the best of this family. Here:
- **other training lengths** for it: 1 and 2 weeks, and 12 weeks (the weather regression of the ETS notebook was
  best on 12 weeks: more days to learn the weather's effect);
- **daily totals** (top-down): AutoARIMA on the day's total — without and with the day's weather (daily means:
  humidity in daylight, wind³, temperature, day length) — spread over the hours by the last 7 days' profile.

Tried and left out (worse than the seasonal naive on the 6 most recent windows): one ARIMAX per hour of the day
(24 daily series with that hour's weather), and DHR with cloud-based interactions (clear-sky curve × (1 − cloud)).

In [12]:
MORE_DHR = {}
for K in (3, 5):
    for c in ("1w", "2w", "12w"):
        MORE_DHR[f"DHR K={K} + weather + sun log, {c}"] = arima(context=CONTEXTS[c], order=None, seasonal_search=False,
                                                               fourier_K={DAY: K}, xcols=X_SETS["weather + sun"])
dhr_table = run("ARIMAX / DHR", MORE_DHR)
print("DHR + weather + sun: day-2 point skill by training length (sections 3 and 4):")
display(grid_view(window_table([n for n, f in FAMILY.items() if f == "ARIMAX / DHR" and "+ weather + sun log" in n
                                and n.startswith("DHR")])).round(3))
show(dhr_table, "DHR on other training lengths (day 2):")

DAILY_ARIMA = {f"daily ARIMA x profile log, {days}d": top_down(lambda: AutoARIMA(), days, 7, "log") for days in (56, 112)}
DAILY_ARIMA.update({f"daily ARIMAX x profile log, {days}d": top_down(lambda: AutoARIMA(), days, 7, "log", X_DAILY)
                    for days in (56, 112)})
show(run("ARIMA top-down", DAILY_ARIMA), "daily totals x hourly profile (day 2):")
elapsed()

ARIMAX / DHR: 6 forecaster(s) x 20 windows


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()


... (195 outputs left out)

  ✓ daily ARIMAX x profile log, 112d               RMSE    278.6  (skill 0.891)   5.5s


daily totals x hourly profile (day 2):


,RMSE skill,MAPE skill,point skill,Winkler skill,coverage,beat bench,seconds
"daily ARIMAX x profile log, 112d",0.873,0.457,0.665,1.359,0.390,9,5.525
"daily ARIMAX x profile log, 56d",0.956,0.422,0.689,1.468,0.396,9,4.906
"daily ARIMA x profile log, 56d",0.880,0.600,0.740,1.010,0.596,14,1.015
"daily ARIMA x profile log, 112d",0.974,0.562,0.768,1.287,0.460,11,0.757
benchmark,1.000,1.000,1.000,1.000,0.817,0,NaN


[31.1 min since the start of Section 2]


## Summary and export

In [13]:
summary, per_window = window_table(per_window=True)
print("day 2 (the private hours):")
display(summary.head(30))
print("all 48 hours:")
display(window_table(part="all").head(15))
print("day-2 RMSE relative to the seasonal naive, window by window (window 1 = the most recent):")
display(per_window.head(15))
for f in dict.fromkeys(FAMILY.values()):
    if f != "simple":
        print(f"best {f}: {best_of(f)}")
export("family_arima")
elapsed()

day 2 (the private hours):


,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
"DHR K=3 + weather + sun log, 2w",266.097,308.735,2.362e+03,0.881,0.793,0.271,0.532,8.620e-01,3,16,209.354
"DHR K=5 + weather + sun log, 2w",283.530,299.117,2.423e+03,0.896,0.845,0.262,0.553,8.844e-01,1,13,22.988
"SARIMA(1,0,1)(2,1,3) + weather + sun log, 4w",274.349,355.589,2.719e+03,0.877,0.817,0.312,0.564,9.922e-01,0,14,NaN
"DHR K=10 + weather + sun log, 4w",247.254,471.036,2.430e+03,0.867,0.736,0.413,0.575,8.866e-01,0,19,NaN
"DHR K=5 + weather + sun log, 4w",240.211,503.682,2.452e+03,0.865,0.715,0.441,0.578,8.948e-01,0,18,NaN
"DHR K=3 + weather + sun log, 4w",245.906,491.152,2.319e+03,0.860,0.732,0.430,0.581,8.463e-01,3,18,NaN
"SARIMA(3,0,0)(2,1,3) + humidity + wind + sun log, 4w",289.403,352.935,5.075e+05,0.923,0.862,0.309,0.586,1.852e+02,0,12,NaN
"SARIMA(1,0,1)(2,1,3) + humidity + wind + sun log, 4w",288.417,358.319,2.764e+03,0.881,0.859,0.314,0.587,1.009e+00,1,13,NaN
"DHR K=10 + weather + sun log, 8w",244.083,514.658,2.469e+03,0.831,0.727,0.451,0.589,9.009e-01,1,16,NaN
"DHR K=5 + weather + sun log, 8w",238.198,538.284,2.473e+03,0.831,0.709,0.472,0.591,9.024e-01,1,18,NaN


all 48 hours:


,RMSE,MAPE,Winkler,coverage,RMSE skill,MAPE skill,point skill,Winkler skill,won,beat bench,seconds
"DHR K=3 + weather + sun log, 2w",279.856,224.627,2466.620,0.889,0.895,0.298,0.596,0.903,1,14,209.354
"DHR K=5 + weather + sun log, 2w",308.252,228.837,2536.300,0.894,0.986,0.304,0.645,0.928,1,11,22.988
"DHR K=3 + weather + sun log, 4w",274.661,328.549,2152.299,0.877,0.878,0.436,0.657,0.788,2,13,NaN
"SARIMA(3,0,0)(2,1,3) + weather + sun bc, 4w",275.321,330.444,2045.051,0.967,0.880,0.439,0.659,0.749,0,13,NaN
"DHR K=5 + weather + sun log, 4w",272.355,338.886,2378.644,0.878,0.871,0.450,0.660,0.871,1,13,NaN
"DHR K=10 + weather + sun log, 4w",283.887,316.428,2375.739,0.882,0.908,0.420,0.664,0.870,1,14,NaN
"DHR K=10 + weather + sun log, 8w",274.186,352.900,2297.804,0.842,0.877,0.468,0.673,0.841,0,11,NaN
"DHR K=3 + weather + sun log, 8w",271.480,362.778,2195.674,0.838,0.868,0.481,0.675,0.804,0,14,NaN
"DHR K=5 + weather + sun log, 8w",270.687,369.957,2292.835,0.841,0.865,0.491,0.678,0.839,0,13,NaN
"SARIMA(1,0,1)(2,1,3) + weather + sun log, 4w",322.510,247.342,2701.606,0.882,1.031,0.328,0.680,0.989,0,11,NaN


day-2 RMSE relative to the seasonal naive, window by window (window 1 = the most recent):


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
"DHR K=3 + weather + sun log, 2w",0.91,0.42,0.55,0.51,0.38,0.47,1.68,0.82,0.40,0.75,0.75,0.56,0.75,0.68,1.23,0.90,0.97,0.44,1.24,2.15
"DHR K=5 + weather + sun log, 2w",1.41,0.69,0.98,0.74,1.47,0.44,1.60,0.80,0.40,0.69,0.77,0.60,0.80,0.73,1.12,1.01,0.77,0.35,1.11,2.51
"SARIMA(1,0,1)(2,1,3) + weather + sun log, 4w",0.94,2.24,1.25,1.36,0.58,0.26,1.31,0.34,0.43,0.79,0.98,0.54,0.83,0.84,0.98,0.87,1.02,0.29,0.85,2.62
"DHR K=10 + weather + sun log, 4w",0.71,0.81,0.39,0.61,0.93,0.63,0.95,0.38,0.44,0.77,0.90,0.57,0.84,0.71,0.90,0.86,0.84,0.28,0.69,3.29
"DHR K=5 + weather + sun log, 4w",0.66,0.78,0.39,0.61,1.30,0.61,0.84,0.41,0.44,0.72,0.83,0.58,0.89,0.72,0.84,0.89,0.78,0.28,0.77,2.54
"DHR K=3 + weather + sun log, 4w",0.75,0.53,0.36,0.66,0.96,0.69,1.10,0.41,0.45,0.78,0.82,0.54,0.75,0.72,0.97,0.84,1.00,0.34,0.89,2.32
"SARIMA(3,0,0)(2,1,3) + humidity + wind + sun log, 4w",1.29,2.46,1.28,1.60,0.95,0.30,0.98,0.30,0.39,0.90,1.00,0.57,0.87,0.98,1.29,1.16,1.11,0.30,0.70,2.60
"SARIMA(1,0,1)(2,1,3) + humidity + wind + sun log, 4w",0.97,2.41,1.50,1.64,0.79,0.24,1.25,0.29,0.39,0.90,1.04,0.57,0.86,0.98,0.98,0.93,1.13,0.30,0.70,2.70
"DHR K=10 + weather + sun log, 8w",0.69,1.27,1.02,1.05,0.76,0.32,0.54,0.38,0.51,0.81,0.82,0.59,0.88,0.79,0.94,0.89,0.86,0.32,0.61,2.61
"DHR K=5 + weather + sun log, 8w",0.52,1.14,0.90,0.89,0.69,0.27,0.54,0.39,0.51,0.80,0.81,0.61,0.91,0.80,0.96,0.91,0.87,0.33,0.62,2.48


... (3 outputs left out)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()
/Library/Frameworks/Python.framework

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsforecast/arima.py:608: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  result = model.fit()


saved family_arima_bt.csv (62 models) and family_arima_final.csv
[31.3 min since the start of Section 2]
